# Imports

In [2]:
import os
import json
import pickle
import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from model import EvacuationModel

# Experiment Setup

In [ ]:
# Create experiments directory structure
def create_experiment_structure():
    """Create directory structure for experiments."""
    base_dir = Path("experiments")
    base_dir.mkdir(exist_ok=True)
    return base_dir

def save_experiment_metadata(exp_dir, model_params, agent_params, description=""):
    """Save experiment configuration and metadata."""
    metadata = {
        "experiment_date": datetime.datetime.now().isoformat(),
        "description": description,
        "model_parameters": model_params,
        "agent_parameters": agent_params
    }
    
    with open(exp_dir / "experiment_config.json", "w") as f:
        json.dump(metadata, f, indent=2)
    
    print(f"Saved experiment metadata to {exp_dir / 'experiment_config.json'}")

def save_experiment_data(exp_dir, model_df, agent_df, final_model):
    """Save simulation results and final model state."""
    # Save dataframes as pickle
    with open(exp_dir / "model_data.pkl", "wb") as f:
        pickle.dump(model_df, f)
    
    with open(exp_dir / "agent_data.pkl", "wb") as f:
        pickle.dump(agent_df, f)
    
    
    print(f"Saved simulation data to {exp_dir}")
    print(f"  - model_data.pkl: {model_df.shape}")
    print(f"  - agent_data.pkl: {agent_df.shape}")
    
def load_experiment_results(exp_dir, load_detailed_data=False, parameter_filter=None):
    """
    Load results from a completed experiment with flexible data access.
    
    Args:
        exp_dir: Path to experiment directory
        load_detailed_data: If True, also load model/agent dataframes for each run
        parameter_filter: Dict to filter runs, e.g., {"v0": 2.0} or {"leader_proportion": 0.1}
    
    Returns:
        results_dict: Dictionary containing all experiment data
    """
    exp_dir = Path(exp_dir)
    
    # Load basic experiment info
    with open(exp_dir / "experiment_config.json", "r") as f:
        config = json.load(f)
    
    # Load summary and detailed results
    summary_df = pd.read_csv(exp_dir / "summary_results.csv")
    detailed_df = pd.read_csv(exp_dir / "detailed_results.csv")
    
    # Load raw results if available
    all_results = None
    if (exp_dir / "all_results.json").exists():
        with open(exp_dir / "all_results.json", "r") as f:
            all_results = json.load(f)
    
    print(f"Loaded experiment: {config['description']}")
    print(f"Date: {config['experiment_date']}")
    print(f"Total runs: {len(detailed_df)}")
    
    # Build results dictionary
    results = {
        "config": config,
        "summary_df": summary_df,
        "detailed_df": detailed_df,
        "all_results": all_results,
        "exp_dir": exp_dir
    }
    
    # Identify the varied parameter from the data
    varied_params = []
    for col in detailed_df.columns:
        if col not in ['run', 'total_time', 'evacuated', 'injured', 'evacuation_rate', 
                      'flow_rate', 'simulation_steps', 'computation_time', 
                      'final_agents_remaining', 'timeout']:
            if detailed_df[col].nunique() > 1:
                varied_params.append(col)
    
    if varied_params:
        print(f"Varied parameter(s): {', '.join(varied_params)}")
        results["varied_parameters"] = varied_params
    
    # Apply parameter filter if specified
    if parameter_filter:
        filtered_df = detailed_df.copy()
        for param, value in parameter_filter.items():
            if param in filtered_df.columns:
                filtered_df = filtered_df[filtered_df[param] == value]
                print(f"Filtered to {param} = {value}: {len(filtered_df)} runs")
            else:
                print(f"Warning: Parameter '{param}' not found in data")
        results["filtered_df"] = filtered_df
    
    # Load detailed simulation data if requested
    if load_detailed_data:
        print(f"Loading detailed simulation data...")
        detailed_data = load_simulation_data(exp_dir, parameter_filter)
        results["simulation_data"] = detailed_data
        print(f"Loaded simulation data for {len(detailed_data)} runs")
    
    return results

def load_simulation_data(exp_dir, parameter_filter=None):
    """
    Load model and agent dataframes for specific runs.
    
    Args:
        exp_dir: Path to experiment directory
        parameter_filter: Dict to filter which runs to load, e.g., {"v0": 2.0}
    
    Returns:
        simulation_data: Dict with run info and dataframes
    """
    exp_dir = Path(exp_dir)
    simulation_data = {}
    
    # Get all run directories
    run_dirs = [d for d in exp_dir.iterdir() if d.is_dir() and d.name.startswith(('v0_', 'leader_', 'param_'))]
    
    for run_dir in run_dirs:
        # Load run configuration
        run_config_file = run_dir / "run_config.json"
        if not run_config_file.exists():
            continue
            
        with open(run_config_file, "r") as f:
            run_config = json.load(f)
        
        # Apply filter if specified
        if parameter_filter:
            skip_run = False
            for param, value in parameter_filter.items():
                # Check in agent_params first, then model_params
                agent_value = run_config.get("agent_params", {}).get(param)
                model_value = run_config.get("model_params", {}).get(param)
                actual_value = run_config.get(param + "_actual")
                
                run_value = agent_value or model_value or actual_value
                
                if run_value != value:
                    skip_run = True
                    break
            
            if skip_run:
                continue
        
        # Load the dataframes
        model_data_file = run_dir / "model_data.pkl"
        agent_data_file = run_dir / "agent_data.pkl"
        
        run_data = {
            "run_config": run_config,
            "run_dir": run_dir,
            "run_name": run_dir.name
        }
        
        if model_data_file.exists():
            with open(model_data_file, "rb") as f:
                run_data["model_df"] = pickle.load(f)
        
        if agent_data_file.exists():
            with open(agent_data_file, "rb") as f:
                run_data["agent_df"] = pickle.load(f)
        
        simulation_data[run_dir.name] = run_data
    
    return simulation_data

# Baseline experiment

In [ ]:
def run_baseline_experiment(v0_values=None, experiment_name="baseline", description="Helbing replication", n_runs=1):
    """
    Run baseline experiment to replicate Helbing's faster-is-slower effect.
    
    Args:
        v0_values: List of desired speeds to test. If None, uses default range.
        experiment_name: Name for this experiment
        description: Description of the experiment
        n_runs: Number of runs per v0 value (for averaging)
    """
    if v0_values is None:
        v0_values = [0.5, 0.8, 1.0, 1.3, 1.5, 1.8, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0]
    
    # Create experiment directory
    base_dir = create_experiment_structure()
    exp_dir = base_dir / f"{experiment_name}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
    exp_dir.mkdir(exist_ok=True)
    
    print(f"Starting {experiment_name} experiment")
    print(f"Results will be saved to: {exp_dir}")
    print(f"Testing {len(v0_values)} different v0 values")
    print(f"Running {n_runs} simulation(s) per v0 value")
    
    # Model parameters (Helbing configuration)
    model_params = {
        "n_agents": 200,
        "width": 15.0,
        "height": 15.0,
        "exit_width": 1.0,
        "num_exits": 1,
        "dt": 0.1,
        "integration_method": "euler",
        "agent_type": "simple",
        "enable_fire": False,  # Disable fire for baseline
        "max_steps": 1000,    # 100 seconds at dt=0.1
        "exit_preset": "center_right"
    }

    base_agent_params = {
        "v0": 1.3,  # Will be overridden per experiment
    }
    
    # Save experiment configuration
    save_experiment_metadata(exp_dir, model_params, base_agent_params, description)
    
    # Results storage
    all_results = []
    
    total_simulations = len(v0_values) * n_runs
    current_sim = 0
    
    for i, v0 in enumerate(v0_values):
        print(f"\n--- Testing v0 = {v0:.1f} m/s ({i+1}/{len(v0_values)}) ---")
        
        v0_results = []  # Store results for this v0 across multiple runs
        
        for run in range(n_runs):
            current_sim += 1
            if n_runs > 1:
                print(f"  Run {run+1}/{n_runs} (Overall: {current_sim}/{total_simulations})")
            
            # Update agent parameters for this run
            agent_params = base_agent_params.copy()
            agent_params["v0"] = v0
            
            # Create model with fixed exit
            model = EvacuationModel(
                agent_parameters=agent_params,
                seed=42 + run,  # Different seed for each run
                **{k: v for k, v in model_params.items() if k not in ["max_steps"]}
            )
            
            # Run simulation
            start_time = datetime.datetime.now()
            
            for step in range(model_params["max_steps"]):
                if step % 2000 == 0 and step > 0:
                    remaining = len([a for a in model.agents if getattr(a, "is_pedestrian", False)])
                    elapsed = step * model.dt
                    print(f"    Step {step} ({elapsed:.1f}s): {remaining} agents remaining")
                
                model.step()
                
                if not model.running:
                    print(f"Completed at step {step} ({step * model.dt:.1f}s)")
                    break
            
            end_time = datetime.datetime.now()
            simulation_duration = (end_time - start_time).total_seconds()
            
            # Get results
            model_df = model.datacollector.get_model_vars_dataframe()
            agent_df = model.datacollector.get_agent_vars_dataframe()
            
            # Calculate metrics
            total_time = len(model_df) * model.dt
            evacuated = model.n_agents - len([a for a in model.agents if getattr(a, "is_pedestrian", False)])
            injured = len([a for a in model.agents if getattr(a, "is_pedestrian", False) and getattr(a, "injured", False)])
            
            # Average flow rate (agents per second)
            if total_time > 0:
                flow_rate = evacuated / total_time
            else:
                flow_rate = 0
            
            result = {
                "v0": v0,
                "run": run,
                "total_time": total_time,
                "evacuated": evacuated,
                "injured": injured,
                "evacuation_rate": evacuated / model.n_agents,
                "flow_rate": flow_rate,
                "simulation_steps": len(model_df),
                "computation_time": simulation_duration,
                "final_agents_remaining": len([a for a in model.agents if getattr(a, "is_pedestrian", False)]),
                "timeout": len(model_df) >= model_params["max_steps"]
            }
            
            v0_results.append(result)
            all_results.append(result)
            
            # Save individual run data
            run_dir = exp_dir / f"v0_{v0:.1f}_run_{run}"
            run_dir.mkdir(exist_ok=True)
            save_experiment_data(run_dir, model_df, agent_df, model)
            
            # Save run-specific parameters
            run_config = {
                "model_params": model_params,
                "agent_params": agent_params,
                "v0_actual": v0,
                "run_number": run,
                "seed": 42 + run
            }
            with open(run_dir / "run_config.json", "w") as f:
                json.dump(run_config, f, indent=2)
        
        # Calculate statistics for this v0 across runs
        if n_runs > 1:
            avg_time = np.mean([r["total_time"] for r in v0_results])
            std_time = np.std([r["total_time"] for r in v0_results])
            avg_flow = np.mean([r["flow_rate"] for r in v0_results])
            
            print(f"v0={v0:.1f}: {avg_time:.1f}±{std_time:.1f}s, flow={avg_flow:.2f} agents/s")
        else:
            print(f"v0={v0:.1f}: {v0_results[0]['total_time']:.1f}s, flow={v0_results[0]['flow_rate']:.2f} agents/s")

    # Save summary results
    results_df = pd.DataFrame(all_results)
    results_df.to_csv(exp_dir / "detailed_results.csv", index=False)
    
    # Create aggregated summary (mean across runs)
    if n_runs > 1:
        summary_stats = []
        for v0 in v0_values:
            v0_data = results_df[results_df['v0'] == v0]
            summary_stats.append({
                'v0': v0,
                'total_time_mean': v0_data['total_time'].mean(),
                'total_time_std': v0_data['total_time'].std(),
                'flow_rate_mean': v0_data['flow_rate'].mean(),
                'flow_rate_std': v0_data['flow_rate'].std(),
                'evacuation_rate_mean': v0_data['evacuation_rate'].mean(),
                'injured_mean': v0_data['injured'].mean(),
                'timeout_rate': v0_data['timeout'].mean(),
                'n_runs': len(v0_data)
            })
        summary_df = pd.DataFrame(summary_stats)
        summary_df.to_csv(exp_dir / "summary_results.csv", index=False)
    else:
        # For single runs, summary is the same as detailed
        summary_df = results_df.copy()
        summary_df.to_csv(exp_dir / "summary_results.csv", index=False)
    
    # Save raw results
    with open(exp_dir / "all_results.json", "w") as f:
        json.dump(all_results, f, indent=2)
    
    print(f"\nExperiment completed!")
    print(f"All results saved to: {exp_dir}")
    print(f"Summary: {len(v0_values)} v0 values × {n_runs} runs = {len(all_results)} total simulations")

    return summary_df, exp_dir, all_results

## Quick test
- here I ran it twice with 1 replication per condition and with 2 to make sure that the code behaves the same and ensure we get the same results given the same seed.

In [ ]:
# Experiment 1: Quick validation (3 v0 values, single run)
print("Experiment 1: Quick Validation")
v0_quick = [1.0, 2.0, 3.0]
results_quick, exp_dir_quick, _ = run_baseline_experiment(
    v0_values=v0_quick,
    experiment_name="quick_validation",
    description="Quick test with 3 v0 values to validate setup",
    n_runs=2
)

### Example1: How to load general experimnet data

In [24]:
results = load_experiment_results("experiments/quick_validation_20251019_132004")

Loaded experiment: Quick test with 3 v0 values to validate setup
Date: 2025-10-19T13:20:04.767505
Total runs: 6
Varied parameter(s): v0


In [25]:
print("Configuration:")
print(results['config'])

print("\nSummary DataFrame:")
display(results['summary_df'])

print("\nDetailed DataFrame:")
display(results['detailed_df'])

print("\nAll Results:")
display(results['all_results'])

Configuration:
{'experiment_date': '2025-10-19T13:20:04.767505', 'description': 'Quick test with 3 v0 values to validate setup', 'model_parameters': {'n_agents': 200, 'width': 15.0, 'height': 15.0, 'exit_width': 1.0, 'num_exits': 1, 'dt': 0.1, 'integration_method': 'euler', 'agent_type': 'simple', 'enable_fire': False, 'max_steps': 1000, 'exit_preset': 'center_right'}, 'agent_parameters': {'v0': 1.3}}

Summary DataFrame:


,v0,total_time_mean,total_time_std,flow_rate_mean,flow_rate_std,evacuation_rate_mean,injured_mean,timeout_rate,n_runs
0,1.0,100.00,0.000000,1.540000,0.042426,0.77,0.0,1.0,2
1,2.0,84.45,3.889087,2.370779,0.109179,1.00,0.0,0.0,2
2,3.0,65.15,2.474874,3.072055,0.116699,1.00,0.0,0.0,2



Detailed DataFrame:


,v0,run,total_time,evacuated,injured,evacuation_rate,flow_rate,simulation_steps,computation_time,final_agents_remaining,timeout
0,1.0,0,100.0,157,0,0.785,1.570000,1000,79.253922,43,True
1,1.0,1,100.0,151,0,0.755,1.510000,1000,101.554780,49,True
2,2.0,0,87.2,200,0,1.000,2.293578,872,60.115927,0,False
3,2.0,1,81.7,200,0,1.000,2.447980,817,52.876207,0,False
4,3.0,0,63.4,200,0,1.000,3.154574,634,47.079608,0,False
5,3.0,1,66.9,200,0,1.000,2.989537,669,52.625414,0,False



All Results:


[{'v0': 1.0,
  'run': 0,
  'total_time': 100.0,
  'evacuated': 157,
  'injured': 0,
  'evacuation_rate': 0.785,
  'flow_rate': 1.57,
  'simulation_steps': 1000,
  'computation_time': 79.253922,
  'final_agents_remaining': 43,
  'timeout': True},
 {'v0': 1.0,
  'run': 1,
  'total_time': 100.0,
  'evacuated': 151,
  'injured': 0,
  'evacuation_rate': 0.755,
  'flow_rate': 1.51,
  'simulation_steps': 1000,
  'computation_time': 101.55478,
  'final_agents_remaining': 49,
  'timeout': True},
 {'v0': 2.0,
  'run': 0,
  'total_time': 87.2,
  'evacuated': 200,
  'injured': 0,
  'evacuation_rate': 1.0,
  'flow_rate': 2.293577981651376,
  'simulation_steps': 872,
  'computation_time': 60.115927,
  'final_agents_remaining': 0,
  'timeout': False},
 {'v0': 2.0,
  'run': 1,
  'total_time': 81.7,
  'evacuated': 200,
  'injured': 0,
  'evacuation_rate': 1.0,
  'flow_rate': 2.4479804161566707,
  'simulation_steps': 817,
  'computation_time': 52.876207,
  'final_agents_remaining': 0,
  'timeout': False

### Example 2: How to load detailed experimnet data

In [27]:

results_v0_2 = load_experiment_results(
    "experiments/quick_validation_20251019_132004",
    load_detailed_data=True,
    parameter_filter={"v0": 2.0}
)

Loaded experiment: Quick test with 3 v0 values to validate setup
Date: 2025-10-19T13:20:04.767505
Total runs: 6
Varied parameter(s): v0
Filtered to v0 = 2.0: 2 runs
Loading detailed simulation data...
Loaded simulation data for 2 runs


In [36]:
# All of the above shown data is still accessible
# If you use a parameter filter: e.g. parameter_filter={"v0": 2.0} - then you can access a filtered_df that shows data for that specific parameter only
display(results_v0_2['filtered_df'])

# In addition if load_detailed_data is True then you could also access the individual model and agent dataframes for specific visualizations
# Access individual run data
run_0_data = results_v0_2['simulation_data']['v0_2.0_run_0']
run_1_data = results_v0_2['simulation_data']['v0_2.0_run_1']

# Get the model dataframes
model_df_run0 = run_0_data['model_df']
model_df_run1 = run_1_data['model_df']

# Get the agent dataframes  
agent_df_run0 = run_0_data['agent_df']
agent_df_run1 = run_1_data['agent_df']

# Print shapes to see the data
print(f"Run 0 - Model data: {model_df_run0.shape}, Agent data: {agent_df_run0.shape}")
print(f"Run 1 - Model data: {model_df_run1.shape}, Agent data: {agent_df_run1.shape}")

display(model_df_run0.head(5))
display(agent_df_run0.head(5))

,v0,run,total_time,evacuated,injured,evacuation_rate,flow_rate,simulation_steps,computation_time,final_agents_remaining,timeout
2,2.0,0,87.2,200,0,1.0,2.293578,872,60.115927,0,False
3,2.0,1,81.7,200,0,1.0,2.447980,817,52.876207,0,False


Run 0 - Model data: (872, 24), Agent data: (74133, 14)
Run 1 - Model data: (817, 24), Agent data: (67863, 14)


,Agents,Average_Speed,Exit_Flow_Total,Exit0_Flow,Exit1_Flow,Exit2_Flow,Exit_Balance_Entropy,Exit0_Pressure,Exit1_Pressure,Exit2_Pressure,...,Injured_Total,Injured_Fire,Injured_Smoke,Injured_Press,Std_v0_init,Std_vmax,Std_radius,Std_mass,Std_vis,Std_panic0
0,200,0.636531,0,0,0,0,0.0,79.575710,0.0,0.0,...,0,0,0,0,0.0,0.0,0.028819,0.0,0.0,0.0
1,200,0.998301,0,0,0,0,0.0,52.909412,0.0,0.0,...,0,0,0,0,0.0,0.0,0.028819,0.0,0.0,0.0
2,200,1.283559,0,0,0,0,0.0,200.783293,0.0,0.0,...,0,0,0,0,0.0,0.0,0.028819,0.0,0.0,0.0
3,200,1.579458,0,0,0,0,0.0,175.510914,0.0,0.0,...,0,0,0,0,0.0,0.0,0.028819,0.0,0.0,0.0
4,199,1.849826,1,1,0,0,-0.0,468.905321,0.0,0.0,...,0,0,0,0,0.0,0.0,0.028819,0.0,0.0,0.0


is_pedestrian  is_leader  injured injury_cause  knows_exit  \
Step AgentID                                                               
1    0                 True      False    False         None       False   
     1                 True      False    False         None       False   
     2                 True      False    False         None       False   
     3                 True      False    False         None       False   
     4                 True      False    False         None       False   

             follow_target_id  smoke_exposure  panic  impatience          x  \
Step AgentID                                                                  
1    0                   None             0.0    0.0         0.0   0.867326   
     1                   None             0.0    0.0         0.0  10.890441   
     2                   None             0.0    0.0         0.0   1.933661   
     3                   None             0.0    0.0         0.0   3.504856   
     4                   None             0.0    0.0         0.0   3.322923   

                     y        vx        vy     Speed  
Step AgentID                                          
1    0        4.375339  0.171753  0.249290  0.302729  
     1        9.984561  0.798444  0.107686  0.805673  
     2        6.389483  2.165178 -0.174230  2.172177  
     3        7.587946 -0.560760  0.129716  0.575568  
     4        9.595591  0.391957 -0.027916  0.392950